In [ ]:
import os
import random
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torchvision
from torchvision import transforms

from sklearn.model_selection import train_test_split

## Classes


In [ ]:
class LatentDataset(Dataset):
    def __init__(self, data_files: list[dict[str, torch.Tensor]]) -> None:
        super().__init__()
        self.X = []
        self.y = []

        for file_data in data_files:
            x = file_data["x"]
            y = file_data["y"]

            for t in range(x.shape[0]):
                self.X.append(x[t])
                self.y.append(y[t])

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[index], self.y[index]

In [ ]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor):
        return self.activation(self.batch_norm(self.conv(x)))

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            padding=1,
        )
        self.conv2 = nn.Conv2d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            padding=1,
        )
        self.batch_norm_1 = nn.BatchNorm2d(channels)
        self.batch_norm_2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor):
        residual = x
        out = F.relu(self.batch_norm_1(self.conv1(x)))
        out = self.batch_norm_2(self.conv2(out))
        return F.relu(out + residual)

In [5]:
class LatentDecoder(nn.Module):
    def __init__(
        self,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(32, 24, kernel_size=3, padding=1),
            ResidualBlock(24),
            nn.Conv2d(24, 16, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)

## Functions


In [ ]:
def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    filepath: str,
    is_best: bool = False,
):
    """Save model checkpoint."""
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    torch.save(checkpoint, filepath)

    if not is_best:
        print(f"Checkpoint saved: {filepath}")

In [ ]:
def validate(
    model: nn.Module, test_loader: DataLoader, loss_fn: nn.Module, device: str
):
    model.eval()

    val_loss = 0

    with torch.no_grad():
        for latent, encoding in test_loader:
            latent = latent.to(device)
            encoding = encoding.to(device)

            pred_encoding = model(latent)

            loss = loss_fn(pred_encoding, encoding)

            val_loss += loss.item()

    model.train()

    return val_loss / len(test_loader)

In [ ]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: str,
    loss_fn: nn.Module | None = None,
    optimizer: torch.optim.Optimizer | None = None,
    scheduler: ReduceLROnPlateau | None = None,
    max_norm: float = 1.0,
    num_epochs: int = 50,
    save_dir: str = "./saved_models",
    print_every: int | None = None,
):
    os.makedirs(save_dir, exist_ok=True)

    optimizer = (
        torch.optim.Adam(model.parameters(), lr=1e-3) if not optimizer else optimizer
    )
    loss_fn = torch.nn.MSELoss() if not loss_fn else loss_fn

    history = {
        "train_loss": [],
        "test_loss": [],
    }

    avg_train_loss = 0
    best_val_loss = float("inf")
    num_batches = len(train_loader)

    print("\n" + "=" * 70)
    print(f"Training Starting - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Epochs: {num_epochs}")
    print("=" * 70 + "\n")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0

        for batch_idx, (latent, encoding) in enumerate(train_loader):
            latent = latent.to(device)
            encoding = encoding.to(device)

            pred_encoding = model(latent)

            loss = loss_fn(pred_encoding, encoding)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
            optimizer.step()

            train_loss += loss.item()
            if print_every is not None:
                if (batch_idx + 1) % print_every == 0:
                    print(
                        f"E[{epoch+1:02d}/{num_epochs}] "
                        f"B[{batch_idx+1:03d}/{num_batches}] | "
                        f"L:{loss.item():.3f}"
                    )

        avg_train_loss = train_loss / num_batches

        val_loss = validate(model, test_loader, loss_fn, device)

        if scheduler:
            scheduler.step(val_loss)

        history['train_loss'].append(avg_train_loss)
        history['test_loss'].append(val_loss)

        print(f"\n{'='*70}")
        print(f"Epoch [{epoch+1}/{num_epochs}] Summary:")
        print(f"{'-'*70}")
        print(f"  Train: L={avg_train_loss:.4f}")
        print(f"  Val: L={val_loss:.4f}")
        print(f"{'='*70}\n")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(
                model,
                optimizer,
                epoch,
                val_loss,
                os.path.join(save_dir, 'best_model.pth'),
                is_best=True,
            )
            print(f"  Best model saved! (Val Loss: {val_loss:.4f})\n")

        torch.cuda.empty_cache()

    save_checkpoint(
        model,
        optimizer,
        num_epochs - 1,
        avg_train_loss,
        os.path.join(save_dir, "final_model.pth"),
    )

    print("\n  Training Complete!")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print("=" * 70 + "\n")

    return history

## Setup


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SOURCE_DIR = "./Final_Model/dataset/dataset_processado/"

TEST_SIZE = 0.2

BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 50
MAX_NORM = 1.0
NUM_EPOCHS = 50

SCHEDULER_PATIENCE = 5
SCHEDULER_FACTOR = 0.5

In [10]:
dataset_files = [
    torch.load(os.path.join(SOURCE_DIR, f)) for f in os.listdir(SOURCE_DIR)
]
random.shuffle(dataset_files)

train_data = dataset_files[int(TEST_SIZE * len(dataset_files)) :]
test_data = dataset_files[: int(TEST_SIZE * len(dataset_files))]

In [11]:
train_dataset = LatentDataset(train_data)
test_dataset = LatentDataset(test_data)

train_dataloader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, BATCH_SIZE, shuffle=True)

## Training


In [12]:
# Defining model
model = LatentDecoder().to(DEVICE)

# Optimizer and scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Higher LR
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# Loss Function
loss_fn = torch.nn.MSELoss()

In [13]:
history = train(
    model,
    train_dataloader,
    test_dataloader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    max_norm=MAX_NORM,
    device=DEVICE,
)


Training Starting - 2026-03-11 20:39:13
Device: mps
Epochs: 50


Epoch [1/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0676
  Val: L=0.0162

  Best model saved! (Val Loss: 0.0162)


Epoch [2/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0143
  Val: L=0.0139

  Best model saved! (Val Loss: 0.0139)


Epoch [3/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0130
  Val: L=0.0128

  Best model saved! (Val Loss: 0.0128)


Epoch [4/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0122
  Val: L=0.0131


Epoch [5/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0117
  Val: L=0.0116

  Best model saved! (Val Loss: 0.0116)


Epoch [6/50] Summary:
----------------------------------------------------------------------
  Train: L=0.0114
  Val: L=0.0124




Next steps involve passing the inputs through the frozen VAE decoder
